## Objectivess 

- Webscrapping the framework description from the websites. 
- Get the framework and lot description and convert them into Vectors.
- Store them into Index in the search service in the azure 
- Build a webpage and ask a question and could generate list of frame works and lots suitable for the questions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import requests
from bs4 import BeautifulSoup
from lxml import html
import os

In [2]:
DB_username = os.getenv("DHW_Username")
DB_password = os.getenv("DHW_Password")
DB_Server = os.getenv('DB_SERVER')

### Reading the framework data 

In [3]:
# Input details for the SQL database
DB_TYPE = "mssql"
DB_USER = DB_username
DB_PWD = DB_password
DB_SERVER = DB_Server
DB_PORT = "1433"
DB_NAME = "PBI"
DB_DRIVER = "SQL Server"
# Connect to the database
conn_string = '{}://{}:{}@{}:{}/{}?driver={}'.format(DB_TYPE, DB_USER, DB_PWD, DB_SERVER, DB_PORT, DB_NAME, DB_DRIVER)

engine = create_engine(conn_string)

conn = engine.connect()

# Read the data from the framework
sql = """select RecordID, Category, SubCategory, Framework, Framework_key, FrameworkNumber, FrameworkLotsID, LotNumber, LotDescription, Status, FrameworkStatus from Frameworks
where FrameworkStatus = 'Live'"""

df_retrieval = pd.read_sql(sql, conn)


In [4]:
No_frameworks = df_retrieval['FrameworkNumber'].unique()
print(f'No of frameworks live from the Datawarehouse: {len(No_frameworks)}')


No of frameworks live from the Datawarehouse: 123


### Webscrapping the framework Description

In [5]:

# FrameWorkWebsite = "https://www.crowncommercial.gov.uk/agreements/RM6200"

# response  = requests.get(FrameWorkWebsite)
# print(response.raise_for_status()) # if it is None, no error

# soup = BeautifulSoup(response.text, 'html.parser')

In [6]:
# tree = html.fromstring(response.content)

# # title
# title = tree.xpath('/html/body/div[4]/main/div[2]/div[1]/h1/text()')[0].strip()

# print(title)

# # Small description
# desc_small = tree.xpath('/html/body/div[4]/main/div[2]/div[1]/div[1]/text()')[0].strip()
# print(desc_small)


In [7]:
# section_headers = soup.find_all('div', class_='govuk-accordion__section-header')
# Headings = []
# for idx, header in enumerate(section_headers):
#     sub_title = header.find('h2').find('span').get_text(strip=True)
#     Headings.append(sub_title)
#     #print(sub_title)

# print(Headings)

In [8]:
# # Description and Benefits

# wysiwyg_div = soup.find_all('div', class_='wysiwyg-content')

# #print(wysiwyg_div[2])

# desc = f'This is the description of the {title} agreement. \n'
# for element in wysiwyg_div[Headings.index('Description')].find_all(['p','li','h4', 'b']):
#     #print(element.get_text(strip=True))
#     desc = desc + element.get_text(strip=True) + '\n'

# benefits = f'These are the benefits of the {title} agreement. \n'
# for element in wysiwyg_div[Headings.index('Benefits')].find_all(['p', 'li', 'h4', 'b']):
#     benefits = benefits + element.get_text(strip=True) + '\n'

In [9]:
# Web scrap along with error handling 
framework_df = pd.DataFrame(columns=['FrameworkNumber', 'Framework', 'title', 'Small_description', 'Description', 'Benefits'])

for idx, framework in enumerate(No_frameworks):
    #print(framework)
    framework_df.loc[idx, 'FrameworkNumber'] = framework
    framework_df.loc[idx, 'Framework'] = df_retrieval.loc[df_retrieval['FrameworkNumber']==framework, 'Framework'].unique()

    url = f'https://www.crowncommercial.gov.uk/agreements/{framework}'
    

    try: 
        response = requests.get(url)
        print(framework, response.raise_for_status()) # if it is None, no error
        soup = BeautifulSoup(response.text, 'html.parser')
        print(idx)

        tree = html.fromstring(response.content)

        # title
        title = tree.xpath('/html/body/div[4]/main/div[2]/div[1]/h1/text()')[0].strip()

        # Small description
        desc_small = tree.xpath('/html/body/div[4]/main/div[2]/div[1]/div[1]/text()')[0].strip()

        # title and small description
        framework_df.loc[idx, 'title'] = title
        framework_df.loc[idx, 'Small_description'] =  desc_small

        # Description and Benefits 
        section_headers = soup.find_all('div', class_='govuk-accordion__section-header')
        Headings = []
        for idx, header in enumerate(section_headers):
            sub_title = header.find('h2').find('span').get_text(strip=True)
            Headings.append(sub_title)
        
        wysiwyg_div = soup.find_all('div', class_='wysiwyg-content')

        desc=''
        desc = f'This is the description of the {title} agreement. \n'
        for element in wysiwyg_div[Headings.index('Description')].find_all(['p','li','h4', 'b']):
            desc = desc + element.get_text(strip=True) + '\n'
        #print(desc)

        benefits =''
        benefits = f'These are the benefits of the {title} agreement. \n'
        for element in wysiwyg_div[Headings.index('Benefits')].find_all(['p', 'li', 'h4', 'b']):
            benefits = benefits + element.get_text(strip=True) + '\n'
        #print(benefits)
        
        framework_df.loc[idx, 'Description'] = desc
        framework_df.loc[idx, 'Benefits'] =  benefits

    except Exception as e:
        print(url)
        #print(f"Error scraping {url}: {e}")

print(framework_df)

framework_df.to_csv(r'C:\Users\Naresh.Sampara\PycharmProjects\P9_FRAMEWORD_recommender\Framework_data.csv', index=False)



https://www.crowncommercial.gov.uk/agreements/RM930
https://www.crowncommercial.gov.uk/agreements/RM990
RM3825 None
2
RM6068 None
3
https://www.crowncommercial.gov.uk/agreements/RM6068
RM6071 None
4
RM6088 None
5
RM6094 None
6
RM6095 None
7
RM6099 None
8
RM6098 None
9
RM6100 None
10
RM6102 None
11
RM6116 None
12
RM1557.14 None
13
https://www.crowncommercial.gov.uk/agreements/RM1557.14
RM6120 None
14
RM6124 None
15
RM6125 None
16
RM6123 None
17
RM6126 None
18
RM6138 None
19
RM6141 None
20
RM6142 None
21
RM3764.3 None
22
RM6148 None
23
RM1043.8 None
24
https://www.crowncommercial.gov.uk/agreements/RM1043.8
RM6157 None
25
RM6163 None
26
RM6165 None
27
RM6167 None
28
https://www.crowncommercial.gov.uk/agreements/RM6167
RM6168 None
29
RM6170 None
30
RM6171 None
31
RM6173 None
32
RM6174 None
33
RM6175 None
34
RM6177 None
35
RM6179 None
36
RM6181 None
37
RM6182 None
38
RM6183 None
39
RM6184 None
40
RM6186 None
41
RM6187 None
42
RM6188 None
43
RM6193 None
44
RM6194 None
45
RM6195 None
46
RM620

In [11]:
print(framework_df.head(4))

  FrameworkNumber                                Framework  \
0           RM930                HMRC Royal Mail Wholesale   
1           RM990  Postal Services - Royal Mail Initiative   
2          RM3825                                 HSCN DPS   
4          RM6071                        Print Marketplace   

                      title  \
0                       NaN   
1                       NaN   
2  HSCN Access Services DPS   
4         Print Marketplace   

                                   Small_description  \
0                                                NaN   
1                                                NaN   
2  Access to the Health and Social Care Network (...   
4  A self-service platform for buying printed mat...   

                                         Description  \
0                                                NaN   
1                                                NaN   
2  This is the description of the Workforce Impro...   
4  This is the description o